# 3. RAG — cAIuldron v2.0

Indexes 7,913 recipes into ChromaDB and provides similarity-based retrieval.

**First run**: builds the index (~2-3 min).  
**Subsequent runs**: loads existing index (~1 sec, skips re-indexing).

In [ ]:
import json
import ast
import hashlib
from typing import List, Dict, Tuple, Optional

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

print('✅ RAG imports OK')

In [ ]:
def _recipe_to_doc(recipe: Dict) -> Tuple[str, Dict]:
    """Convert a recipe dict to (embed_text, metadata).

    Embed only: title + cuisine + ingredient tags.
    Embedding full instructions makes similarity noisy.
    """
    # ingredient_tags may be stored as string repr of list: "['beef', 'onion']"
    raw_tags = recipe.get('ingredient_tags', '[]')
    try:
        tags = ast.literal_eval(raw_tags) if isinstance(raw_tags, str) else raw_tags
    except Exception:
        tags = []
    ingredients_text = ', '.join(tags) if tags else recipe.get('ingredient', '')

    embed_text = (
        f"Recipe: {recipe.get('recipe_title', 'Unknown')}. "
        f"Cuisine: {recipe.get('cuisine', 'american')}. "
        f"Main ingredients: {ingredients_text}."
    )

    # Instructions preview for context display (not embedded)
    instructions = recipe.get('instructions', [])
    if isinstance(instructions, str):
        instructions = [instructions]
    steps_preview = '; '.join(str(s) for s in instructions[:3])[:500]

    metadata = {
        'title':                recipe.get('recipe_title', 'Unknown'),
        'cuisine':              recipe.get('cuisine', 'american'),
        'difficulty':           recipe.get('difficulty', 'easy'),
        'cooking_time_minutes': int(recipe.get('cooking_time_minutes', 30) or 30),
        'servings':             int(recipe.get('servings', 4) or 4),
        'ingredient_tags':      ingredients_text,
        'instructions_preview': steps_preview,
    }
    return embed_text, metadata

print('✅ Recipe→document converter ready')

In [ ]:
def build_or_load_index(force_rebuild: bool = False) -> chromadb.Collection:
    """Build ChromaDB index from recipes JSON, or load existing one.

    Persists to CHROMA_DIR. Re-index only if collection is missing or corrupt.
    """
    CHROMA_DIR.mkdir(parents=True, exist_ok=True)

    embed_fn = SentenceTransformerEmbeddingFunction(
        model_name=EMBEDDING_MODEL,
        device='cpu',
    )
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))

    # Try loading existing collection
    if not force_rebuild:
        try:
            col = client.get_collection(name=CHROMA_COLLECTION, embedding_function=embed_fn)
            count = col.count()
            if count >= 7000:
                print(f'✅ ChromaDB loaded: {count} recipes (skipping rebuild)')
                return col
            print(f'⚠️  Collection has only {count} docs — rebuilding...')
            client.delete_collection(CHROMA_COLLECTION)
        except Exception:
            pass  # Collection does not exist yet

    # Load recipes JSON
    print(f'📂 Loading recipes from {RECIPES_JSON}...')
    with open(RECIPES_JSON, 'r', encoding='utf-8') as f:
        recipes = json.load(f)
    print(f'   Found {len(recipes)} recipes — building ChromaDB index...')

    col = client.create_collection(
        name=CHROMA_COLLECTION,
        embedding_function=embed_fn,
        metadata={'hnsw:space': 'cosine'},
    )

    BATCH_SIZE = 500
    ids, docs, metas = [], [], []

    for i, recipe in enumerate(recipes):
        doc_text, meta = _recipe_to_doc(recipe)
        title_hash = hashlib.md5(recipe.get('recipe_title', str(i)).encode()).hexdigest()[:8]
        ids.append(f'recipe_{i}_{title_hash}')
        docs.append(doc_text)
        metas.append(meta)

        if len(ids) >= BATCH_SIZE:
            col.add(ids=ids, documents=docs, metadatas=metas)
            ids, docs, metas = [], [], []
            if (i + 1) % 2000 == 0:
                print(f'   Indexed {i + 1}/{len(recipes)}...')

    if ids:  # flush last batch
        col.add(ids=ids, documents=docs, metadatas=metas)

    print(f'✅ ChromaDB index built: {col.count()} recipes')
    return col

# Build / load on notebook run
_rag_collection = build_or_load_index()
print(f'✅ RAG collection ready: "{CHROMA_COLLECTION}"')

In [ ]:
def retrieve_similar_recipes(ingredients: List[str]) -> Tuple[List[Dict], str]:
    """Query ChromaDB for top-K most similar recipes given detected ingredients.

    Returns (recipe_dicts, formatted_context_block_for_llm).
    """
    # Mirror the embedding format used during indexing
    query_text = f"Main ingredients: {', '.join(ingredients)}."

    results = _rag_collection.query(
        query_texts=[query_text],
        n_results=RAG_TOP_K,
        include=['metadatas', 'distances'],
    )

    retrieved = []
    context_lines = ['**Similar Recipes from Our Database:**']

    for i in range(len(results['ids'][0])):
        distance   = results['distances'][0][i]
        similarity = round(1 - distance, 3)  # cosine distance → similarity

        if similarity < RAG_MIN_SIMILARITY:
            continue

        meta = results['metadatas'][0][i]
        recipe_dict = {
            'title':                meta['title'],
            'cuisine':              meta['cuisine'],
            'difficulty':           meta['difficulty'],
            'cooking_time_minutes': meta['cooking_time_minutes'],
            'servings':             meta['servings'],
            'ingredients':          meta['ingredient_tags'],
            'instructions_preview': meta['instructions_preview'],
            'similarity_score':     similarity,
        }
        retrieved.append(recipe_dict)

        context_lines.append(
            f"\n**{i+1}. {meta['title']}** ({meta['cuisine']}, {meta['difficulty']}, "
            f"{meta['cooking_time_minutes']} min, similarity: {similarity})\n"
            f"Ingredients: {meta['ingredient_tags']}\n"
            f"Steps preview: {meta['instructions_preview']}"
        )

    if not retrieved:
        context_lines.append('_No closely matching recipes found in database._')

    return retrieved, '\n'.join(context_lines)

print('✅ retrieve_similar_recipes() ready')

# Quick test
_test_results, _test_ctx = retrieve_similar_recipes(['chicken', 'garlic', 'lemon'])
print(f'   Test query returned {len(_test_results)} results')
if _test_results:
    print(f'   Top result: "{_test_results[0]["title"]}" (similarity: {_test_results[0]["similarity_score"]})')